# Homework 3 — Classification

2026 cohort.  
Dataset: course_lead_scoring_2026.csv

# Setup and getting the data

Use the 2026 lead-scoring dataset, `course_lead_scoring_2026.csv`.

The target is `converted`: whether the client signed up to the platform.

# Data preparation

* Check whether the features have missing values.
* Categorical features: replace missing values with `'NA'`.
* Numerical features: replace missing values with `0.0`.

In [44]:
import pandas as pd
import numpy as np

df = pd.read_csv("../cohorts/2026/data/course_lead_scoring_2026.csv")
df.head()


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [45]:
categorical = ["lead_source", "industry", "employment_status", "location"]
df_cat = df[categorical].fillna("NA")
df_cat.head()

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NA
4,referral,manufacturing,student,north_america


In [46]:
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]
df_num = df[numerical].fillna(0.0)
df_num.head()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,88160.0,3,4,0.64
1,72688.0,3,7,0.72
2,44697.0,3,4,0.58
3,0.0,3,6,0.57
4,24062.0,4,5,0.62


# Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

In [47]:
df_cat["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

technology 

# Question 2

What are the two numerical features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above.

In [48]:
corr = df_num.corr().abs()
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
corr.stack().loc[pairs].sort_values(ascending=False)

interaction_count         lead_score           0.915746
number_of_courses_viewed  lead_score           0.757204
                          interaction_count    0.721609
annual_income             interaction_count    0.122842
dtype: float64

interaction_count` and `lead_score`

# Split the data

Split with `test_size=0.2` and then `test_size=0.25`, both with `random_state=42`.

Keep `converted` out of the feature dataframes.

In [49]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

del df_train["converted"]
del df_val["converted"]
del df_test["converted"]


# Question 3

Which categorical variable has the biggest mutual information score with `converted`? Use the training set only, and round each score to 2 decimals.

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [50]:
from sklearn.metrics import mutual_info_score

In [51]:
df_train[categorical] = df_train[categorical].fillna("NA")

for col in categorical:
    score = mutual_info_score(df_train[col], y_train)
    print(col, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


# Question 4

What accuracy does logistic regression get on the validation set? Round to 2 decimal digits.

Fit with `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)` and one-hot encode the categorical variables.

- 0.55
- 0.65
- 0.75
- 0.85

In [52]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df_train[categorical] = df_train[categorical].fillna("NA")
df_val[categorical] = df_val[categorical].fillna("NA")
df_train[numerical] = df_train[numerical].fillna(0.0)
df_val[numerical] = df_val[numerical].fillna(0.0)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[categorical + numerical].to_dict(orient="records"))
X_val = dv.transform(df_val[categorical + numerical].to_dict(orient="records"))

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

accuracy = (model.predict(X_val) == y_val).mean()
round(accuracy, 2)

/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


np.float64(0.64)

# Question 5

Which feature has the smallest difference when removed? Train as in Question 4, without rounding, then drop one feature at a time.

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

The difference does not have to be positive.

In [53]:
model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

original_accuracy = (model.predict(X_val) == y_val).mean()
original_accuracy

/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


np.float64(0.645)

In [54]:
features = ["lead_source", "number_of_courses_viewed", "interaction_count"]

for feature in features:
    train_small = df_train.drop(columns=[feature])
    val_small = df_val.drop(columns=[feature])

    dv_small = DictVectorizer(sparse=False)
    X_train_small = dv_small.fit_transform(train_small.to_dict(orient="records"))
    X_val_small = dv_small.transform(val_small.to_dict(orient="records"))

    model_small = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    model_small.fit(X_train_small, y_train)

    accuracy_small = (model_small.predict(X_val_small) == y_val).mean()
    difference = original_accuracy - accuracy_small
    print(feature, difference)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.1

# Question 6

Which `C` gives the best validation accuracy? Try `[0.000001, 0.00001, 0.0001, 0.001]`, use all features as in Question 4, and round accuracy to 3 decimal digits.

- 0.000001
- 0.00001
- 0.0001
- 0.001

If several values tie, select the smallest `C`.

In [55]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df_train[categorical] = df_train[categorical].fillna("NA")
df_val[categorical] = df_val[categorical].fillna("NA")
df_train[numerical] = df_train[numerical].fillna(0.0)
df_val[numerical] = df_val[numerical].fillna(0.0)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[categorical + numerical].to_dict(orient="records"))
X_val = dv.transform(df_val[categorical + numerical].to_dict(orient="records"))

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver="liblinear", C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    accuracy = (model.predict(X_val) == y_val).mean()
    print(c, round(accuracy, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tom/.pyenv/versions/3.10.13/lib/python3.1